# NanoBEIR Pooling Sweep Analysis (Relative to Pool Factor 1)

This notebook compares 4 pooling methods:

- `span`
- `hierarchical`
- `kmeans`
- `kmeans_sk` (kmeans + sklearn backend)

Relative performance is anchored at **pool factor = 1**, defined as **100%** for each method.

In [ ]:
from __future__ import annotations

import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid", context="talk")
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 220)

## Parameters

Edit this cell to control methods, pool factors, datasets, and metric.

In [ ]:
# Paths
REPO_ROOT = Path.cwd().resolve().parents[2] if Path.cwd().name == "evaluation" else Path.cwd().resolve()
EVAL_DIR = REPO_ROOT / "examples" / "pooling" / "evaluation"
FULL_ROOT = EVAL_DIR / "results" / "nanobeir_full"
KMEANS_SK_ROOT = EVAL_DIR / "results" / "nanobeir_full_kmeans_sk"

# Data selection
TARGET_MODEL = "mixedbread-ai/mxbai-edge-colbert-v0-32m"
ALL_METHODS = ["span", "hierarchical", "kmeans", "kmeans_sk"]
METHODS_TO_INCLUDE = ["span", "hierarchical", "kmeans", "kmeans_sk"]
POOL_FACTORS_TO_INCLUDE = [1, 2, 3, 4, 5, 7, 10, 15, 20]  # or None for all
DATASETS_TO_INCLUDE = None  # Example: ["SCIDOCS", "SciFact"]; None means all
ANCHOR_POOL_FACTOR = 1

# Metric selection
KEY_METRICS = ["ndcg@10", "mrr@10", "map@100", "recall@10", "recall@100", "accuracy@10"]
PRIMARY_METRIC = "ndcg@10"

assert PRIMARY_METRIC in KEY_METRICS
assert ANCHOR_POOL_FACTOR >= 1
assert set(METHODS_TO_INCLUDE).issubset(set(ALL_METHODS))

In [ ]:
def find_latest_sweep_dir(root: Path, prefix: str = "nanobeir_pooling_sweep") -> Path:
    if not root.exists():
        raise FileNotFoundError(f"Missing root: {root}")
    candidates = sorted(
        [p for p in root.iterdir() if p.is_dir() and p.name.startswith(prefix + "_")],
        key=lambda p: p.name,
    )
    if not candidates:
        raise FileNotFoundError(f"No sweep dirs found under {root}")
    return candidates[-1]


def parse_scores(scores: dict[str, float]) -> tuple[dict[str, dict[str, float]], dict[str, float]]:
    dataset_results: dict[str, dict[str, float]] = {}
    mean_results: dict[str, float] = {}

    for key, value in scores.items():
        if key.startswith("NanoBEIR_mean_MaxSim_"):
            metric = key.replace("NanoBEIR_mean_MaxSim_", "")
            mean_results[metric] = float(value)
            continue

        if not key.startswith("Nano") or "_MaxSim_" not in key or "mean" in key.lower():
            continue

        left, metric = key.split("_MaxSim_", 1)
        dataset_name = left.replace("Nano", "", 1)
        dataset_results.setdefault(dataset_name, {})[metric] = float(value)

    return dataset_results, mean_results


def canonical_method(pool_method: str, use_sklearn: bool) -> str:
    if pool_method == "kmeans" and use_sklearn:
        return "kmeans_sk"
    return pool_method


def load_sweep_results(run_dir: Path) -> tuple[pd.DataFrame, pd.DataFrame]:
    mean_rows = []
    ds_rows = []

    for json_path in sorted(run_dir.glob("nanobeir_*.json")):
        if json_path.name == "overview_summary.json":
            continue

        payload = json.loads(json_path.read_text())
        if payload.get("model") != TARGET_MODEL:
            continue

        method = canonical_method(payload.get("pool_method"), bool(payload.get("use_sklearn", False)))
        if method not in ALL_METHODS:
            continue

        pool_factor = int(payload["pool_factor"])
        eval_time_s = float(payload.get("evaluation_time_seconds", np.nan))

        dataset_results, mean_results = parse_scores(payload.get("scores", {}))

        mean_rows.append({
            "run_dir": run_dir.name,
            "results_json": json_path.name,
            "method": method,
            "pool_factor": pool_factor,
            "evaluation_time_seconds": eval_time_s,
            **{f"mean_{m}": mean_results.get(m, np.nan) for m in KEY_METRICS},
        })

        for dataset_name, metrics in dataset_results.items():
            ds_rows.append({
                "run_dir": run_dir.name,
                "results_json": json_path.name,
                "method": method,
                "pool_factor": pool_factor,
                "dataset": dataset_name,
                **{m: metrics.get(m, np.nan) for m in KEY_METRICS},
            })

    mean_df = pd.DataFrame(mean_rows)
    ds_df = pd.DataFrame(ds_rows)

    if not mean_df.empty:
        mean_df = mean_df.sort_values(["method", "pool_factor"]).drop_duplicates(["method", "pool_factor"], keep="last").reset_index(drop=True)
    if not ds_df.empty:
        ds_df = ds_df.sort_values(["dataset", "method", "pool_factor"]).drop_duplicates(["dataset", "method", "pool_factor"], keep="last").reset_index(drop=True)

    return mean_df, ds_df


def apply_filters(mean_df: pd.DataFrame, ds_df: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    out_mean = mean_df.copy()
    out_ds = ds_df.copy()

    out_mean = out_mean[out_mean["method"].isin(METHODS_TO_INCLUDE)]
    out_ds = out_ds[out_ds["method"].isin(METHODS_TO_INCLUDE)]

    if POOL_FACTORS_TO_INCLUDE is not None:
        out_mean = out_mean[out_mean["pool_factor"].isin(POOL_FACTORS_TO_INCLUDE)]
        out_ds = out_ds[out_ds["pool_factor"].isin(POOL_FACTORS_TO_INCLUDE)]

    if DATASETS_TO_INCLUDE is not None:
        selected = set(DATASETS_TO_INCLUDE)
        out_ds = out_ds[out_ds["dataset"].isin(selected)]

    out_mean = out_mean.sort_values(["method", "pool_factor"]).reset_index(drop=True)
    out_ds = out_ds.sort_values(["dataset", "method", "pool_factor"]).reset_index(drop=True)
    return out_mean, out_ds


def add_relative_columns_overall(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for metric in KEY_METRICS:
        metric_col = f"mean_{metric}"
        rel_col = f"rel_{metric}_pct"
        delta_col = f"delta_{metric}_pct"

        anchors = (
            out[out["pool_factor"] == ANCHOR_POOL_FACTOR][["method", metric_col]]
            .rename(columns={metric_col: "anchor_value"})
            .drop_duplicates(subset=["method"], keep="last")
        )

        out = out.merge(anchors, on="method", how="left")
        out[rel_col] = 100.0 * out[metric_col] / out["anchor_value"]
        out[delta_col] = out[rel_col] - 100.0
        out = out.drop(columns=["anchor_value"])

    return out


def add_relative_columns_dataset(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for metric in KEY_METRICS:
        rel_col = f"rel_{metric}_pct"
        delta_col = f"delta_{metric}_pct"

        anchors = (
            out[out["pool_factor"] == ANCHOR_POOL_FACTOR][["dataset", "method", metric]]
            .rename(columns={metric: "anchor_value"})
            .drop_duplicates(subset=["dataset", "method"], keep="last")
        )

        out = out.merge(anchors, on=["dataset", "method"], how="left")
        out[rel_col] = 100.0 * out[metric] / out["anchor_value"]
        out[delta_col] = out[rel_col] - 100.0
        out = out.drop(columns=["anchor_value"])

    return out

In [ ]:
latest_full = find_latest_sweep_dir(FULL_ROOT)
latest_kmeans_sk = find_latest_sweep_dir(KMEANS_SK_ROOT)

print("Latest full sweep:", latest_full)
print("Latest kmeans_sk sweep:", latest_kmeans_sk)

mean_full, ds_full = load_sweep_results(latest_full)
mean_sk, ds_sk = load_sweep_results(latest_kmeans_sk)

mean_df_raw = pd.concat([mean_full, mean_sk], ignore_index=True)
ds_df_raw = pd.concat([ds_full, ds_sk], ignore_index=True)

mean_df, ds_df = apply_filters(mean_df_raw, ds_df_raw)

if mean_df.empty or ds_df.empty:
    raise ValueError("No rows left after filtering. Check parameter selections.")

mean_rel_df = add_relative_columns_overall(mean_df)
ds_rel_df = add_relative_columns_dataset(ds_df)

print("Rows (overall):", len(mean_rel_df))
print("Rows (per-dataset):", len(ds_rel_df))
print("Methods:", sorted(mean_rel_df["method"].unique().tolist()))
print("Pool factors:", sorted(mean_rel_df["pool_factor"].unique().tolist()))
print("Datasets:", sorted(ds_rel_df["dataset"].unique().tolist()))

missing_overall_anchor = sorted(set(mean_rel_df.loc[mean_rel_df[f"rel_{PRIMARY_METRIC}_pct"].isna(), "method"].unique()))
if missing_overall_anchor:
    print("Warning: missing anchor pool factor for methods:", missing_overall_anchor)

missing_dataset_anchor = sorted(set(ds_rel_df.loc[ds_rel_df[f"rel_{PRIMARY_METRIC}_pct"].isna(), "dataset"].unique()))
if missing_dataset_anchor:
    print("Warning: missing dataset-method anchors for datasets:", missing_dataset_anchor)

In [ ]:
# Optional export for downstream use
out_dir = EVAL_DIR / "results"
out_dir.mkdir(parents=True, exist_ok=True)

mean_abs_csv = out_dir / "analysis_overview_mean_metrics_long.csv"
mean_rel_csv = out_dir / "analysis_overview_mean_metrics_long_relative.csv"
ds_abs_csv = out_dir / "analysis_overview_per_dataset_metrics_long.csv"
ds_rel_csv = out_dir / "analysis_overview_per_dataset_metrics_long_relative.csv"

mean_df.to_csv(mean_abs_csv, index=False)
mean_rel_df.to_csv(mean_rel_csv, index=False)
ds_df.to_csv(ds_abs_csv, index=False)
ds_rel_df.to_csv(ds_rel_csv, index=False)

print("Saved:")
print(" -", mean_abs_csv)
print(" -", mean_rel_csv)
print(" -", ds_abs_csv)
print(" -", ds_rel_csv)

## Overall Relative Performance

Anchor definition: for each method, metric at pool factor `1` is `100%`.

In [ ]:
rel_metric_col = f"rel_{PRIMARY_METRIC}_pct"
delta_metric_col = f"delta_{PRIMARY_METRIC}_pct"
abs_metric_col = f"mean_{PRIMARY_METRIC}"

overall_relative_summary = (
    mean_rel_df.groupby("method", as_index=False)
    .agg(
        runs=("pool_factor", "count"),
        best_pool_factor=(rel_metric_col, lambda s: int(mean_rel_df.loc[s.idxmax(), "pool_factor"])),
        best_relative_pct=(rel_metric_col, "max"),
        best_relative_delta_pct=(delta_metric_col, "max"),
        avg_relative_pct=(rel_metric_col, "mean"),
        avg_relative_delta_pct=(delta_metric_col, "mean"),
        best_abs_metric=(abs_metric_col, "max"),
        avg_eval_time_sec=("evaluation_time_seconds", "mean"),
    )
    .sort_values("best_relative_pct", ascending=False)
    .reset_index(drop=True)
)

overall_relative_summary

In [ ]:
overall_relative_pivot = (
    mean_rel_df.pivot(index="pool_factor", columns="method", values=rel_metric_col)
    .reindex(columns=METHODS_TO_INCLUDE)
    .sort_index()
)

overall_relative_pivot

In [ ]:
plt.figure(figsize=(11, 6))
for method in METHODS_TO_INCLUDE:
    part = mean_rel_df[mean_rel_df["method"] == method].sort_values("pool_factor")
    if part.empty:
        continue
    plt.plot(part["pool_factor"], part[rel_metric_col], marker="o", linewidth=2, label=method)

plt.axhline(100.0, color="black", linestyle="--", linewidth=1)
plt.title(f"Overall Relative {PRIMARY_METRIC} (Anchor: pool={ANCHOR_POOL_FACTOR} => 100%)")
plt.xlabel("Pool Factor")
plt.ylabel(f"Relative {PRIMARY_METRIC} (%)")
plt.xticks(sorted(mean_rel_df["pool_factor"].unique()))
plt.legend(title="Method")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(11, 6))
for method in METHODS_TO_INCLUDE:
    part = mean_rel_df[mean_rel_df["method"] == method].sort_values("pool_factor")
    if part.empty:
        continue
    plt.plot(part["pool_factor"], part[delta_metric_col], marker="o", linewidth=2, label=method)

plt.axhline(0.0, color="black", linestyle="--", linewidth=1)
plt.title(f"Overall Delta {PRIMARY_METRIC} vs Anchor (percentage points)")
plt.xlabel("Pool Factor")
plt.ylabel(f"Delta {PRIMARY_METRIC} (% points)")
plt.xticks(sorted(mean_rel_df["pool_factor"].unique()))
plt.legend(title="Method")
plt.tight_layout()
plt.show()

## Per-Dataset Relative Performance

In [ ]:
def plot_dataset_relative(dataset: str, metric: str = PRIMARY_METRIC, show_delta: bool = False):
    if metric not in KEY_METRICS:
        raise ValueError(f"Unknown metric: {metric}. Choose from: {KEY_METRICS}")

    rel_col = f"rel_{metric}_pct"
    delta_col = f"delta_{metric}_pct"

    part = ds_rel_df[ds_rel_df["dataset"] == dataset].sort_values(["method", "pool_factor"])
    if part.empty:
        raise ValueError(f"No data for dataset: {dataset}")

    y_col = delta_col if show_delta else rel_col

    plt.figure(figsize=(11, 6))
    for method in METHODS_TO_INCLUDE:
        mpart = part[part["method"] == method].sort_values("pool_factor")
        if mpart.empty:
            continue
        plt.plot(mpart["pool_factor"], mpart[y_col], marker="o", linewidth=2, label=method)

    baseline = 0.0 if show_delta else 100.0
    plt.axhline(baseline, color="black", linestyle="--", linewidth=1)
    suffix = "delta (% points)" if show_delta else "relative (%)"
    plt.title(f"{dataset}: {metric} {suffix} vs pool factor")
    plt.xlabel("Pool Factor")
    plt.ylabel(f"{metric} {suffix}")
    plt.xticks(sorted(part["pool_factor"].unique()))
    plt.legend(title="Method")
    plt.tight_layout()
    plt.show()

In [ ]:
# Example usage
plot_dataset_relative("SCIDOCS", metric=PRIMARY_METRIC, show_delta=False)

In [ ]:
# Small multiples: relative PRIMARY_METRIC across datasets
datasets = sorted(ds_rel_df["dataset"].unique())
num = len(datasets)
cols = 3
rows = int(np.ceil(num / cols))

fig, axes = plt.subplots(rows, cols, figsize=(18, 4.2 * rows), sharex=True, sharey=False)
axes = np.array(axes).reshape(-1)

for ax, dataset in zip(axes, datasets):
    part = ds_rel_df[ds_rel_df["dataset"] == dataset].sort_values(["method", "pool_factor"])
    for method in METHODS_TO_INCLUDE:
        mpart = part[part["method"] == method].sort_values("pool_factor")
        if mpart.empty:
            continue
        ax.plot(mpart["pool_factor"], mpart[rel_metric_col], marker="o", linewidth=1.8, label=method)
    ax.axhline(100.0, color="black", linestyle="--", linewidth=0.9)
    ax.set_title(dataset)
    ax.set_xlabel("Pool")
    ax.set_ylabel("Rel %")

for ax in axes[num:]:
    ax.axis("off")

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="upper center", ncol=max(1, len(METHODS_TO_INCLUDE)), frameon=True)
fig.suptitle(f"Per-Dataset Relative {PRIMARY_METRIC} Curves (pool={ANCHOR_POOL_FACTOR} => 100%)", y=1.02)
fig.tight_layout()
plt.show()